# 強化学習アルゴリズムの整理

強化学習の手法は、「行動して、結果を見て、次の行動を少しずつ直す」という共通の流れから整理できます。価値関数やベルマン更新は、TD 法、Q 学習、SARSA、n-step TD、TD(λ)、深層強化学習で形を変えながら使われます。


## 比較軸を揃える

強化学習のアルゴリズム名は、「何を予想しているのか」「その予想をどう直すのか」という設計の組み合わせを表しています。

- 何を学ぶか: その状態の良さ `V(s)` か、その状態でその行動を取る良さ `Q(s,a)` か
- どこまで先の報酬を見るか: すぐ次だけで直すか、数手先までまとめて見るか
- 目標値をどう作るか: いちばん良さそうな行動を仮定するか、実際に選んだ行動を使うか
- 誤差をどこへ配るか: いまの1手だけ直すか、過去の行動にも少しずつ配るか
- 値をどう持つか: 小さい表で持つか、大きい問題なので関数で近似するか


## アルゴリズム対応表

| 手法 | 学ぶ量 | 目標値の作り方 | 行動方針との関係 | 主な論点 |
|---|---|---|---|---|
| TD法 | `V(s)` | `r + \gamma V(s')` | 方策を評価する寄り | 予想を次の状態で少しずつ直す最小形 |
| Q学習 | `Q(s,a)` | `r + \gamma \max_{a'} Q(s', a')` | off-policy | 次は最善を選ぶと仮定して強気に更新する |
| SARSA | `Q(s,a)` | `r + \gamma Q(s', a')` | on-policy | 実際に探索しながら取った行動込みで学ぶ |
| n-step TD法 | `V` または `Q` | n-step return | 手法依存 | すぐ先だけでなく数手先まで見て直す |
| TD(λ) | 主に `V` | λ-return / trace | 手法依存 | 1-step と Monte Carlo の間を連続的につなぐ |
| Eligibility Trace | `V` や `Q` | TD 誤差を過去へ配る | 手法依存 | どの過去行動に功績や責任を戻すか |
| 深層強化学習 | 近似された `Q` や policy | 手法依存 | 手法依存 | 表で持てない大きな問題を関数で扱う |


## 同じ経験から、目標値だけを差し替えて見る

強化学習の式は名前ごとに別物に見えますが、多くの場合は「いまの予測」と「新しく作った目標値」の差を使って値を直しています。違いが出るのは、目標値をどの材料で作るかです。まずは同じ状態遷移を用意し、Monte Carlo、TD、Q 学習、SARSA がどこで分かれるかを数値で比べます。

In [ ]:
gamma = 0.9
alpha = 0.4

# ある状態 s0 で行動 a_go を取り、報酬 r を受けて s1 へ進んだとします。
r = 0.2
V = {'s0': 0.30, 's1': 0.60}
Q = {
    ('s0', 'a_go'): 0.30,
    ('s1', 'safe'): 0.50,
    ('s1', 'risky'): 0.90,
}
actual_next_action = 'safe'
future_rewards = [0.2, 0.0, 1.0]

mc_target = sum((gamma ** i) * reward for i, reward in enumerate(future_rewards))
td_target = r + gamma * V['s1']
q_learning_target = r + gamma * max(Q[('s1', a)] for a in ['safe', 'risky'])
sarsa_target = r + gamma * Q[('s1', actual_next_action)]

for name, target in [
    ('Monte Carlo', mc_target),
    ('TD(0)', td_target),
    ('Q-learning', q_learning_target),
    ('SARSA', sarsa_target),
]:
    old_value = Q[('s0', 'a_go')]
    updated = old_value + alpha * (target - old_value)
    print(f'{name:13s} target={target:.3f} updated Q={updated:.3f}')

出力を見ると、同じ経験を使っていても目標値が変わるだけで更新後の値が変わります。Monte Carlo は実際に後で得た報酬列を足し切ります。TD(0) は次状態の価値 `V(s1)` を借ります。Q 学習は次に最善行動を取れると仮定し、SARSA は実際に選んだ `safe` の価値を使います。この差が、強気に最適行動を学ぶのか、探索込みの現在の方策を学ぶのかを分けます。

## 設計軸で分類する

何を学ぶか、何歩先まで見るか、次の行動をどう扱うかを分けると、各手法の違いを同じ表で比較できます。次のコードでは、手法名を特徴量の表として持ち、違いを機械的に並べます。


In [ ]:
algorithms = [
    {'name': 'TD(0)', 'value': 'V', 'horizon': '1-step', 'next_action': 'policy evaluation'},
    {'name': 'Q-learning', 'value': 'Q', 'horizon': '1-step', 'next_action': 'max'},
    {'name': 'SARSA', 'value': 'Q', 'horizon': '1-step', 'next_action': 'sampled action'},
    {'name': 'n-step TD', 'value': 'V/Q', 'horizon': 'n-step', 'next_action': 'bootstrap after n'},
    {'name': 'TD(lambda)', 'value': 'V/Q', 'horizon': 'mixed', 'next_action': 'weighted returns'},
]
for item in algorithms:
    print(f"{item['name']:<12} value={item['value']:<3} horizon={item['horizon']:<8} target={item['next_action']}")

こうして並べると、手法名よりも設計上の差分が見えます。Q 学習と SARSA はどちらも 1-step の `Q` 更新ですが、次の行動価値を最大値で見るか、実際に選んだ行動で見るかが違います。


## どういう順で違いを見ると混乱しにくいか

最初に詰まりやすいのは、式が違うからではなく、どこが同じでどこだけ差し替わっているかが見えなくなるからです。実際にはかなりの部分が共通です。

1. まず「この状態や行動はどれくらい将来の報酬につながりそうか」を予想する。
2. その予想を、すぐ次の結果だけで直すか、数手先まで見て直すかを選ぶ。
3. 次の行動としていちばん良いものを仮定するか、実際に取った行動を使うかで Q学習 と SARSA が分かれる。
4. ずれをいまの1手だけ直すか、過去にも少し戻して配るかで trace 系へ進む。
5. 表では扱えない大きな状態空間になったら、ニューラルネットなどの関数近似へ進む。


## 使い分けの見取り図

- まず「次の状態を手がかりに予想を直す」感覚を理解したい: TD法
- 次は最善を選べると仮定して、理想的な行動価値を学びたい: Q学習
- 探索で少し危ない行動も取りうる現実の方策をそのまま改善したい: SARSA
- 遠い報酬をもっと早く学習へ入れたい: n-step TD
- 1-step と Monte Carlo の中間を滑らかに動かしたい: TD(λ)
- 成功や失敗の原因を現在だけでなく少し前の行動にも戻したい: Eligibility Trace
- 盤面や画像のように状態が大きすぎて表を持てない: 深層強化学習


## 目標値の違いを更新方向として見る

target の値だけでなく、古い予測から見て上げるのか下げるのかを確認すると、アルゴリズムの性格が分かります。同じ経験でも、楽観的な target なら値を強く上げ、保守的な target なら小さく直します。


In [ ]:
old = 0.30
targets = {
    'TD(0)': td_target,
    'Q-learning': q_learning_target,
    'SARSA': sarsa_target,
    'Monte Carlo': mc_target,
}
for name, target in targets.items():
    direction = 'up' if target > old else 'down'
    delta = target - old
    print(f'{name:13s} direction={direction:<4} delta={delta:+.3f}')

更新方向を見れば、「どの手法が強気に値を上げるか」「どの手法が探索込みで控えめに直すか」が分かります。式を読むときも、target が何で作られ、古い予測との差がどちらを向くかを追います。


## 誤差をどこまで戻すかも、別の設計軸になる

次に、目標値の作り方ではなく、更新の影響をどこまで戻すかを見ます。1-step TD は現在の状態だけを強く直します。Eligibility trace は、直近に訪れた状態ほど大きく、古い状態ほど小さく同じ TD 誤差を配ります。

In [ ]:
gamma = 0.9
lam = 0.7
alpha = 0.2
states = ['s0', 's1', 's0', 's2']
V = {'s0': 0.2, 's1': 0.4, 's2': 0.1}
trace = {s: 0.0 for s in V}

td_error = 0.8
for s in states:
    for key in trace:
        trace[key] *= gamma * lam
    trace[s] += 1.0

for s in V:
    V[s] += alpha * td_error * trace[s]

print('trace =', {k: round(v, 3) for k, v in trace.items()})
print('updated V =', {k: round(v, 3) for k, v in V.items()})

同じ TD 誤差でも、trace を使うと「いま」だけでなく、直前に関わった状態にも更新が戻ります。これが TD(λ) と Eligibility Trace 系の直感です。強化学習の各アルゴリズムは、目標値をどう作るか、誤差をどこへ配るか、値を表で持つか関数で近似するか、という少数の設計軸で整理できます。

## 残したいこと

アルゴリズム名だけでは、手法同士の関係は見えにくくなります。強化学習は、「行動して結果を受け取り、そのずれで次の予想を直す」という共通骨格の上で、目標値の作り方と誤差の配り方を変えています。この整理によって、新しい手法でも共通部分と差分を分けられます。
